# Entrenamiento en Kaggle (modelos clásicos, sin redes neuronales)

Antes de correr:
1. No hace falta **Add Input**: el dataset combinado (`FRANKENSTEIN` en la celda Parámetros) se descarga solo. Si lo adjuntas, se usa el adjunto.
2. **Settings** → Internet: **On** (requiere teléfono verificado). No hace falta GPU: todo corre en CPU.
3. Para correr sin tener la pestaña abierta: **Save Version → Save & Run All**.
4. **Add-ons → Secrets** → activa `GITHUB_TOKEN` para que el modelo se publique solo al final (ver docs/detalles.md, "Publicar el modelo"). Sin él, el notebook entrena igual pero no publica.
5. Las fotos del equipo se bajan solas de la carpeta de Google Drive (`EQUIPO_DRIVE` en la celda de parámetros).
6. Los parámetros de los modelos se ajustan en la celda **Parámetros**, sin tocar el repo.

Este notebook solo clona el repo y llama a `src/`. Los cambios de código van en el repo, no aquí.

In [ ]:
import os, subprocess

REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    r = subprocess.run(["git", "clone", "https://github.com/11909/Clasificador-fruta-verdura.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo. Activa Settings → Internet en el panel derecho "
                           "(Kaggle pide verificar tu teléfono) y vuelve a correr esta celda.\n" + r.stderr)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"])  # trae la última versión de main
%cd {REPO}
!pip install -q -r requirements.txt

## Parámetros

Todo lo que se puede ajustar sin tocar el repo. Cambia los valores y corre de aquí hacia abajo.

- **Versión preliminar:** con `MUESTRA_POR_ETIQUETA` (por defecto 5000) entrena con el mismo número de frutas y de verduras tomadas de todo lo disponible, dividido 80/20. Ponlo en `None` para usar los splits normales cuando ya haya fotos del equipo.
- **Datos:** `MAX_POR_CLASE` más alto usa más imágenes (mejor, pero más lento).
- **Limpieza:** `LIMPIEZA` quita los cuadros casi repetidos de Fruits-360 y deja como máximo 100 fotos de esa fuente por clase, para que no domine ni meta ruido. Se aplica al cargar; el dataset de Kaggle no cambia. `None` la apaga.
- **Características y validación:** `FEATURES` elige qué se mide de cada foto; quita tipos para comparar su aporte. `CV` elige cómo se mide el desempeño: `uno_fuera_grupo` (leave-one-out por alimento, por defecto) entrena ~140 veces por modelo dejando fuera un alimento cada vez, y dice qué tan bien reconoce alimentos que nunca vio. Para una prueba rápida usa `grupos`.
- **Modelos:** cada entrada de `MODELOS` se compara con validación cruzada y se queda el de mayor f1_macro. Quita las que no quieras o cambia sus `params` (son los de scikit-learn). Opciones de `name`: logistic_regression, svm, knn, naive_bayes, decision_tree, random_forest, gradient_boosting, voting, stacking. `pca` reduce dimensiones antes del modelo (entero = componentes, decimal = varianza a conservar). `grid` prueba varias combinaciones con GridSearchCV.

In [ ]:
# --- Versión preliminar ---
# Número entero: junta todas las imágenes (Kaggle y Drive), toma esa cantidad de frutas y la misma de
# verduras (repartidas entre clases) y las divide 80/20 en train y test. None = usar los splits del
# dataset combinado y los topes por clase de abajo.
MUESTRA_POR_ETIQUETA = 5000    # p. ej. 5000 frutas + 5000 verduras; súbelo para usar más (más lento)
FRACCION_TEST_MUESTRA = 0.2

# --- Datos ---
FRANKENSTEIN = "mendezrosalesmiguel/frankeinstein"   # dataset combinado; se descarga solo si no está en Add Input
EQUIPO_DRIVE = "https://drive.google.com/drive/folders/1l4Q_VRv__PdaO16_aKlgEwdX50k4FT3S"  # "" = sin fotos del equipo
FRACCION_TEST_EQUIPO = 0.3     # parte de las fotos del equipo que va a test
MAX_POR_CLASE = 150            # imágenes por alimento para entrenar (None = todas; más = más lento)
MAX_POR_CLASE_VERDURA = 180    # hay ~62 clases de verdura y ~75 de fruta: así quedan ~11,000 de cada una
MAX_POR_CLASE_TEST = None      # None = todo el conjunto de prueba
BALANCEAR_ETIQUETAS = True     # mismo número de frutas que de verduras
EXCLUIR_GRUPOS = []            # ["nueces_semillas"] para sacar nueces y semillas
# Limpieza de Fruits-360 al cargar (el dataset de Kaggle no cambia). None = sin limpiar.
#   umbral_duplicado: quita cuadros casi iguales de la misma clase (más alto = quita más; None = no quitar)
#   tope_fruits360: máximo de fotos de Fruits-360 por clase (None = sin tope)
#   quitar_fondo_blanco: True descarta fotos de estudio de cualquier fuente (saca todo Fruits-360)
LIMPIEZA = {"umbral_duplicado": 6, "tope_fruits360": 100, "quitar_fondo_blanco": False}
MINIMO_EQUIPO = 80             # fotos reales del equipo que pide como mínimo cada alimento del plan

# --- Características y validación ---
# tipos: color (histograma HSV), hog (forma), lbp (textura) son los de antes. Los nuevos:
#   color_espacial (color por zonas de la imagen), hog_fino (HOG con celdas más chicas),
#   lbp_multi (textura a 3 escalas), forma (área, redondez y alargamiento de la silueta)
FEATURES = {"tamano": 64, "tipos": ["color", "color_espacial", "hog_fino", "lbp_multi", "forma"], "bins_color": 16}
# tipo de validación cruzada:
#   "uno_fuera_grupo": leave-one-out por alimento; cada vuelta deja fuera un alimento completo (~140 vueltas)
#   "grupos": 5 vueltas sin repetir alimento entre entrenamiento y validación (más rápido)
#   "estratificada": 5 vueltas al azar (sale más alto porque valida con alimentos ya vistos)
#   "uno_fuera": leave-one-out clásico, una imagen por vuelta; solo con MUESTRA_POR_ETIQUETA <= 300
CV = {"tipo": "uno_fuera_grupo", "folds": 5}   # folds se usa en "grupos", "estratificada" y en los grid
AJUSTAR_UMBRAL = True                 # busca el mejor umbral de verdura en vez de 0.5
SEMILLA = 42

# --- Modelos a comparar ---
MODELOS = [
    {"name": "logistic_regression", "params": {"C": 1.0}},
    {"name": "knn", "params": {"n_neighbors": 7}, "pca": 0.95},
    {"name": "naive_bayes"},
    {"name": "random_forest", "params": {"n_estimators": 300}, "pca": 0.95},
    {"name": "gradient_boosting", "params": {"max_iter": 300, "learning_rate": 0.1}, "pca": 0.95},
    {"name": "svm", "params": {"kernel": "rbf", "C": 10, "gamma": "scale"}, "pca": 0.95},
    {"name": "voting", "pca": 0.95, "params": {"estimadores": {
        "svm": {"kernel": "rbf", "C": 10, "gamma": "scale"},
        "random_forest": {"n_estimators": 300},
        "gradient_boosting": {"max_iter": 300},
    }}},
    # Ejemplo con búsqueda de hiperparámetros:
    # {"name": "random_forest", "grid": {"randomforestclassifier__max_depth": [None, 20],
    #                                    "randomforestclassifier__n_estimators": [200, 400]}},
]

Localiza el dataset combinado de Kaggle, baja las fotos del equipo de Drive y genera la config de esta corrida con los parámetros de arriba.

In [ ]:
import glob, os, yaml

# Dataset combinado del equipo (tiene manifest.csv): el de Add Input o, si no está, se descarga.
manifest = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
if not manifest and FRANKENSTEIN:
    try:
        import kagglehub
        ruta = kagglehub.dataset_download(FRANKENSTEIN)
        manifest = glob.glob(f"{ruta}/**/manifest.csv", recursive=True)
    except Exception as e:
        print(f"No pude descargar {FRANKENSTEIN} ({e}). Pide a Miguel Angel acceso al dataset.")
if manifest:
    base = os.path.dirname(manifest[0])
    train_dirs, test_dirs = [f"{base}/train"], [f"{base}/test"]
else:
    train_dirs = sorted(glob.glob("/kaggle/input/**/Training", recursive=True))
    test_dirs = sorted(glob.glob("/kaggle/input/**/Test", recursive=True))
if not train_dirs or not test_dirs:
    print("Contenido de /kaggle/input:", glob.glob("/kaggle/input/*/*")[:20])
    raise FileNotFoundError("No encontré las carpetas de imágenes. Activa Settings → Internet o adjunta "
                            "el dataset combinado con Add Input y vuelve a correr.")
print("Kaggle train:", train_dirs[0])
print("Kaggle test:", test_dirs[0])

# Fotos del equipo desde Google Drive
equipo_dirs = []
if EQUIPO_DRIVE:
    from src.equipo import descargar_drive
    try:
        equipo_dirs = descargar_drive(EQUIPO_DRIVE)
    except Exception as e:
        # Sin Drive se entrena igual, solo con el dataset combinado
        print(f"AVISO: no pude bajar la carpeta de Drive ({type(e).__name__}: {e}).\n"
              "Sigo SIN las fotos del equipo. Revisa que esté compartida como 'Cualquier persona con el enlace' "
              "y que Settings → Internet esté activo; si Drive dice que hubo demasiados accesos, espera un rato.")
    print("Fotos del equipo:", equipo_dirs)
else:
    print("Sin fotos del equipo (EQUIPO_DRIVE vacío).")

cfg = yaml.safe_load(open("configs/baseline.yaml"))
cfg["data"].update({
    "train_dirs": train_dirs[:1], "test_dirs": test_dirs[:1], "equipo_dir": equipo_dirs or None,
    "fraccion_test_equipo": FRACCION_TEST_EQUIPO, "max_por_clase": MAX_POR_CLASE,
    "max_por_clase_verdura": MAX_POR_CLASE_VERDURA,
    "max_por_clase_test": MAX_POR_CLASE_TEST, "balancear_etiquetas": BALANCEAR_ETIQUETAS,
    "excluir_grupos": EXCLUIR_GRUPOS, "limpieza": LIMPIEZA,
    "muestra_por_etiqueta": MUESTRA_POR_ETIQUETA, "fraccion_test_muestra": FRACCION_TEST_MUESTRA,
})
cfg.update({"features": FEATURES, "cv": CV, "ajustar_umbral": AJUSTAR_UMBRAL, "seed": SEMILLA, "models": MODELOS})
yaml.safe_dump(cfg, open("configs/run.yaml", "w"), allow_unicode=True)

Cuenta las imágenes de cada fuente y escribe `faltantes.txt` (en `/kaggle/working/outputs/`): cuántas fotos le faltan al equipo por alimento para que cada uno llegue a `MAX_POR_CLASE` en train (`MAX_POR_CLASE_VERDURA` en las verduras), con al menos `MINIMO_EQUIPO` fotos reales. También revisa que todas las carpetas tengan etiqueta en `data/mapping.csv`.

In [ ]:
from pathlib import Path
from src.dataset import dividir_equipo, listar_muestras
from src.equipo import contar_por_alimento, escribir_faltantes, faltantes, leer_plan

kaggle = contar_por_alimento(listar_muestras(train_dirs[:1], excluir_grupos=EXCLUIR_GRUPOS))
equipo_muestras = []
for carpeta in equipo_dirs:
    a, b = dividir_equipo(carpeta, FRACCION_TEST_EQUIPO, SEMILLA, EXCLUIR_GRUPOS)
    equipo_muestras += a + b
equipo_conteo = contar_por_alimento(equipo_muestras)
print(f"Kaggle: {sum(kaggle.values())} imágenes de train en {len(kaggle)} alimentos")
print(f"Equipo: {len(equipo_muestras)} fotos en {len(equipo_conteo)} alimentos")

salida = Path(cfg["output_dir"]); salida.mkdir(parents=True, exist_ok=True)
filas = faltantes(kaggle, equipo_conteo, MAX_POR_CLASE or 150, FRACCION_TEST_EQUIPO, MINIMO_EQUIPO, leer_plan(),
                  MAX_POR_CLASE_VERDURA)
print(escribir_faltantes(filas, salida / "faltantes.txt"))

Extrae características, compara los modelos con validación cruzada, reentrena el mejor y lo evalúa en test.

In [ ]:
!python -m src.train --config configs/run.yaml

Resultados: tabla de métricas de cada modelo en validación cruzada (`resultados_cv.csv`) y en test (`resultados_test.csv`), y las métricas en test del modelo elegido (`test_metrics.json`). Todo queda en `/kaggle/working/outputs/`.

- **accuracy:** fotos bien clasificadas. **accuracy balanceada:** promedio del acierto en fruta y en verdura.
- **precision de una clase:** de las fotos que el modelo dijo que eran de esa clase, cuántas lo eran.
- **recall de una clase:** de las fotos que eran de esa clase, cuántas encontró.
- **F1:** combina precision y recall. Las columnas `_macro` promedian fruta y verdura.
- En test, cada modelo se entrena con todo train y usa umbral 0.5; el modelo elegido además usa su umbral ajustado (`test_metrics.json`).

In [ ]:
import json
from pathlib import Path
import pandas as pd

SALIDA = Path("/kaggle/working/outputs")
COLUMNAS = ["accuracy", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro",
            "precision_fruta", "recall_fruta", "f1_fruta", "precision_verdura", "recall_verdura", "f1_verdura"]

cv = pd.read_csv(SALIDA / "resultados_cv.csv").set_index("modelo").sort_values("f1_macro", ascending=False)
print("Validación cruzada")
display(cv[COLUMNAS + ["f1_macro_std", "segundos"]].round(3))
if (SALIDA / "resultados_test.csv").exists():
    test_modelos = pd.read_csv(SALIDA / "resultados_test.csv").set_index("modelo").loc[cv.index]
    print("Test (cada modelo, umbral 0.5)")
    display(test_modelos[COLUMNAS].round(3))
print("Modelo elegido en test:")
print(json.dumps(json.load(open(SALIDA / "test_metrics.json")), indent=2))

Gráficas: métricas de validación cruzada de cada modelo (la línea es ± desviación estándar de f1_macro), tiempo de entrenamiento y matriz de confusión en test del modelo elegido. Se guardan en `outputs/comparacion_modelos.png`.

In [ ]:
import csv, json
import matplotlib.pyplot as plt
import numpy as np

AZUL, NARANJA, AQUA = "#2a78d6", "#eb6834", "#1baf7a"   # paleta categórica validada (orden fijo)
TEXTO, TEXTO_2, REJILLA = "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"axes.edgecolor": REJILLA, "axes.labelcolor": TEXTO_2, "xtick.color": TEXTO_2,
                     "ytick.color": TEXTO, "axes.spines.top": False, "axes.spines.right": False})

filas = list(csv.DictReader(open("/kaggle/working/outputs/resultados_cv.csv")))
filas.sort(key=lambda f: float(f["f1_macro"]))           # el mejor queda arriba
modelos = [f["modelo"] for f in filas]
y = np.arange(len(modelos))
test = json.load(open("/kaggle/working/outputs/test_metrics.json"))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, max(3.5, 0.55 * len(modelos) + 1.5)),
                                    gridspec_kw={"width_ratios": [2.2, 1.2, 1]})

# 1. Métricas de validación cruzada (puntos: las diferencias suelen ser pequeñas)
for k, (met, color, nombre) in enumerate([("accuracy", AZUL, "accuracy"),
                                          ("balanced_accuracy", NARANJA, "accuracy balanceada"),
                                          ("f1_macro", AQUA, "f1_macro")]):
    v = np.array([float(f[met]) for f in filas])
    yy = y + (k - 1) * 0.22
    if met == "f1_macro":
        err = np.array([float(f["f1_macro_std"]) for f in filas])
        ax1.hlines(yy, v - err, v + err, color=color, linewidth=2, alpha=0.5)
    ax1.scatter(v, yy, s=70, color=color, edgecolor="white", linewidth=2, zorder=3, label=nombre)
mejor = filas[-1]
ax1.annotate(f"{float(mejor['f1_macro']):.3f}", (float(mejor["f1_macro"]) + float(mejor["f1_macro_std"]), y[-1] + 0.22),
             xytext=(8, 0), textcoords="offset points", va="center", color=TEXTO, fontsize=9)
ax1.set_yticks(y, modelos)
ax1.set_xlabel("puntaje en validación cruzada")
ax1.grid(axis="x", color=REJILLA, linewidth=0.8)
ax1.set_axisbelow(True)
ax1.legend(loc="lower right", frameon=False)
ax1.set_title("Métricas por modelo", loc="left", color=TEXTO)

# 2. Tiempo de validación cruzada
seg = [float(f["segundos"]) for f in filas]
ax2.barh(y, seg, color=AZUL, height=0.6)
for yi, s in zip(y, seg):
    ax2.text(s, yi, f" {s:.0f} s", va="center", color=TEXTO_2, fontsize=9)
ax2.set_yticks(y, [""] * len(y))
ax2.tick_params(axis="y", length=0)
ax2.set_xlabel("segundos")
ax2.grid(axis="x", color=REJILLA, linewidth=0.8)
ax2.set_axisbelow(True)
ax2.set_title("Tiempo", loc="left", color=TEXTO)

# 3. Matriz de confusión en test del modelo elegido
m = np.array(test["matriz_confusion"])
ax3.imshow(m, cmap=plt.matplotlib.colors.LinearSegmentedColormap.from_list("azul", ["#cde2fb", "#0d366b"]))
for (r, c_), n in np.ndenumerate(m):
    ax3.text(c_, r, n, ha="center", va="center", color="white" if n > m.max() / 2 else TEXTO, fontsize=12)
ax3.set_xticks([0, 1], ["fruta", "verdura"]); ax3.set_yticks([0, 1], ["fruta", "verdura"])
ax3.set_xlabel("predicho"); ax3.set_ylabel("real")
ax3.spines[:].set_visible(False)
ax3.set_title(f"Test: {test['modelo']}\nacc. balanceada {test['balanced_accuracy']:.3f}", loc="left", color=TEXTO)

fig.tight_layout()
fig.savefig("/kaggle/working/outputs/comparacion_modelos.png", dpi=150, bbox_inches="tight")
plt.show()

Gráficas de todas las métricas por modelo, en validación cruzada y en test (`outputs/metricas_modelos.png`), y la matriz de confusión en test de cada modelo (`outputs/matrices_modelos.png`). En las matrices, cada fila suma 100 %: la diagonal es el recall de cada clase.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

TEXTO, TEXTO_2 = "#0b0b0b", "#52514e"
AZULES = LinearSegmentedColormap.from_list("azul", ["#eef5fe", "#cde2fb", "#2a78d6", "#0d366b"])
ETIQUETAS_COL = ["accuracy", "acc. balanceada", "precision\nmacro", "recall\nmacro", "F1\nmacro",
                 "precision\nfruta", "recall\nfruta", "F1\nfruta", "precision\nverdura", "recall\nverdura", "F1\nverdura"]

tablas = [("Validación cruzada", cv[COLUMNAS])]
if (SALIDA / "resultados_test.csv").exists():
    tablas.append(("Test (umbral 0.5)", test_modelos[COLUMNAS]))
valores = np.concatenate([t.to_numpy().ravel() for _, t in tablas])
vmin, vmax = max(0, np.nanmin(valores) - 0.05), 1

# 1. Todas las métricas por modelo (el mejor en validación cruzada arriba)
fig, ejes = plt.subplots(len(tablas), 1, figsize=(13, (0.5 * len(cv) + 1.6) * len(tablas)), squeeze=False)
for ax, (titulo, tabla) in zip(ejes[:, 0], tablas):
    m = tabla.to_numpy(dtype=float)
    ax.imshow(m, cmap=AZULES, vmin=vmin, vmax=vmax, aspect="auto")
    for (r, c), v in np.ndenumerate(m):
        ax.text(c, r, f"{v:.3f}", ha="center", va="center", fontsize=9,
                color="white" if (v - vmin) / (vmax - vmin) > 0.55 else TEXTO)
    # Resalta el mejor valor de cada columna
    for c in range(m.shape[1]):
        r = int(np.nanargmax(m[:, c]))
        ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, edgecolor="#eb6834", linewidth=2))
    ax.set_xticks(range(len(COLUMNAS)), ETIQUETAS_COL, fontsize=9, color=TEXTO_2)
    ax.set_yticks(range(len(tabla)), tabla.index, color=TEXTO)
    ax.tick_params(length=0)
    ax.xaxis.tick_top()
    ax.axvline(4.5, color="white", linewidth=3); ax.axvline(7.5, color="white", linewidth=3)
    ax.spines[:].set_visible(False)
    ax.set_title(titulo, loc="left", color=TEXTO, pad=34)
fig.text(0.01, 0.0, "Recuadro naranja: el mejor modelo en esa métrica.", color=TEXTO_2, fontsize=9)
fig.tight_layout()
fig.savefig(SALIDA / "metricas_modelos.png", dpi=150, bbox_inches="tight")
plt.show()

# 2. Matriz de confusión en test de cada modelo
if len(tablas) > 1:
    n = len(test_modelos)
    columnas = min(4, n)
    filas_fig = int(np.ceil(n / columnas))
    fig, ejes = plt.subplots(filas_fig, columnas, figsize=(3.4 * columnas, 3.2 * filas_fig), squeeze=False)
    for ax in ejes.ravel()[n:]:
        ax.axis("off")
    for ax, (nombre, fila) in zip(ejes.ravel(), test_modelos.iterrows()):
        cuentas = np.array(json.loads(fila["matriz_confusion"]))
        pct = cuentas / cuentas.sum(axis=1, keepdims=True).clip(min=1)
        ax.imshow(pct, cmap=AZULES, vmin=0, vmax=1)
        for (r, c), p in np.ndenumerate(pct):
            ax.text(c, r, f"{p:.0%}\n({cuentas[r, c]})", ha="center", va="center", fontsize=10,
                    color="white" if p > 0.55 else TEXTO)
        ax.set_xticks([0, 1], ["fruta", "verdura"]); ax.set_yticks([0, 1], ["fruta", "verdura"])
        ax.set_xlabel("predicho", color=TEXTO_2); ax.set_ylabel("real", color=TEXTO_2)
        ax.tick_params(length=0)
        ax.spines[:].set_visible(False)
        ax.set_title(f"{nombre}\nacc. balanceada {fila['balanced_accuracy']:.3f}", loc="left", color=TEXTO, fontsize=10)
    fig.tight_layout()
    fig.savefig(SALIDA / "matrices_modelos.png", dpi=150, bbox_inches="tight")
    plt.show()

## Revisar y publicar el modelo

Prueba rápida del modelo recién entrenado con las entradas de ejemplo. Después, la última celda lo publica sola como Release de GitHub, siempre que supere en accuracy balanceada de test al modelo publicado. Desde ese momento `notebooks/predecir.ipynb`, `python -m src.predict` y la interfaz usan el modelo nuevo.

In [ ]:
from src.clasificador import Clasificador

clf = Clasificador("/kaggle/working/outputs/modelo.joblib")
print(clf.descripcion)
for r in clf.clasificar_lote("entradas/ejemplos.txt"):
    print(r.get("etiqueta", "ERROR"), f"{r.get('confianza', 0):.0%}", r["entrada"].split("/")[-2:], r.get("error", ""))

In [ ]:
import os
from src.publicar import publicar

PUBLICAR = True   # False para entrenar sin publicar
FORZAR = False    # True publica aunque no supere al modelo publicado

token = os.environ.get("GITHUB_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None

if not PUBLICAR:
    print("Publicación desactivada (PUBLICAR = False).")
elif not token:
    print("No encontré el secreto GITHUB_TOKEN, así que el modelo no se publicó. "
          "Agrégalo en Add-ons → Secrets (ver docs/detalles.md, \"Publicar el modelo\") y corre esta celda otra vez.")
else:
    publicar("/kaggle/working/outputs", token, forzar=FORZAR)